# Testar o modelo LoRA registrado no MLflow

Este notebook demonstra como consumir uma versão registrada do adapter LoRA da EscutIA. O adapter é baixado do MLflow Model Registry, combinado com o modelo-base e utilizado para realizar inferências.

> O item registrado é um adapter LoRA, não um modelo completo. Por isso, o modelo-base continua sendo necessário para a inferência.

Esta é uma etapa de disponibilização e consumo do modelo. Nenhum novo treinamento é executado aqui.

## Objetivo

Ao final, teremos validado o fluxo:

`Registered Model no MLflow → download do adapter → modelo-base + LoRA → inferência`

Esse fluxo representa como o adapter pode ser recuperado posteriormente para ser utilizado em um notebook, script ou aplicação como uma API.

In [8]:
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'configs').exists():
    candidato = BASE_DIR / 'EscutIA' / 'fine_tuning_lora'
    if (candidato / 'configs').exists():
        BASE_DIR = candidato

if not (BASE_DIR / 'configs').exists():
    raise FileNotFoundError('Execute o notebook a partir da raiz do projeto ou da pasta fine_tuning_lora.')

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
REGISTERED_MODEL = 'EscutIA-LoRA'
MODEL_VERSION = '1'
MODEL_URI = f'models:/{REGISTERED_MODEL}/{MODEL_VERSION}'
MLFLOW_DB_PATH = (BASE_DIR.parent / 'mlflow.db').resolve()
TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)

cliente = MlflowClient(tracking_uri=TRACKING_URI)
versao = cliente.get_model_version(REGISTERED_MODEL, MODEL_VERSION)

print(f'Modelo registrado: {REGISTERED_MODEL}')
print(f'Versão selecionada: {MODEL_VERSION}')
print(f'URI do modelo: {MODEL_URI}')
print(f'Run de origem: {versao.run_id}')
print(f'Fonte do artefato: {versao.source}')

Modelo registrado: EscutIA-LoRA
Versão selecionada: 1
URI do modelo: models:/EscutIA-LoRA/1
Run de origem: 6cbabc48c6ee40ed94a73e1dfdd7178e
Fonte do artefato: runs:/6cbabc48c6ee40ed94a73e1dfdd7178e/adapter_lora


## 1. Baixar o adapter registrado

A URI `models:/EscutIA-LoRA/1` identifica a versão do modelo no Registry. O MLflow resolve essa referência e baixa os arquivos do adapter para um diretório local de cache.

In [9]:
import mlflow.artifacts

ADAPTER_DIR = Path(
    mlflow.artifacts.download_artifacts(artifact_uri=MODEL_URI)
).resolve()

if not (ADAPTER_DIR / 'adapter_config.json').exists():
    candidatos = list(ADAPTER_DIR.rglob('adapter_config.json'))
    if not candidatos:
        raise FileNotFoundError(f'O artefato baixado não contém adapter_config.json: {ADAPTER_DIR}')
    ADAPTER_DIR = candidatos[0].parent

arquivos_adapter = sorted(caminho.name for caminho in ADAPTER_DIR.iterdir() if caminho.is_file())
print(f'Adapter baixado em: {ADAPTER_DIR}')
print('Arquivos encontrados:', arquivos_adapter)

Adapter baixado em: C:\Users\mdbaa\AppData\Local\Temp\tmpa5hig45w
Arquivos encontrados: ['adapter_config.json', 'adapter_model.safetensors', 'all_results.json', 'eval_results.json', 'tokenizer.json', 'tokenizer_config.json', 'train_results.json', 'trainer_state.json']


## 2. Carregar o modelo-base com o adapter LoRA

O modelo-base é carregado pelo Transformers. Em seguida, o PEFT aplica o adapter recuperado do MLflow. O dispositivo é selecionado automaticamente entre CUDA, XPU, MPS e CPU.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if getattr(torch, 'cuda', None) is not None and torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif getattr(torch, 'xpu', None) is not None and torch.xpu.is_available():
    DEVICE = torch.device('xpu')
    DTYPE = torch.bfloat16
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE = torch.float32
else:
    DEVICE = torch.device('cpu')
    DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=DTYPE,
)
modelo = PeftModel.from_pretrained(modelo_base, ADAPTER_DIR)
modelo = modelo.to(DEVICE)
modelo.eval()

print(f'Dispositivo: {DEVICE}')
print(f'Tipo numérico: {DTYPE}')
print('Modelo-base + adapter registrado carregados com sucesso.')

Dispositivo: xpu
Tipo numérico: torch.bfloat16
Modelo-base + adapter registrado carregados com sucesso.


## 3. Executar uma inferência

A função abaixo reproduz o contrato do roteador: receber um texto e responder somente com um JSON contendo o campo `sentimento`.

In [11]:
import json
import re

INSTRUCTION = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo e responda somente com um JSON válido no formato {"sentimento":"<rotulo>"}.'
SYSTEM_PROMPT = 'Você é um roteador de sentimentos. Responda somente com JSON válido no formato {"sentimento":"negativo|neutro|positivo"}.'
LABELS = ('negativo', 'neutro', 'positivo')

def extrair_json(resposta: str):
    trecho = re.search(r'\{.*?\}', resposta, flags=re.DOTALL)
    if trecho is None:
        return None
    try:
        candidato = json.loads(trecho.group(0))
    except json.JSONDecodeError:
        return None
    if set(candidato) != {'sentimento'} or candidato['sentimento'] not in LABELS:
        return None
    return candidato

def classificar_sentimento(texto: str, max_new_tokens: int = 24):
    mensagens = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{INSTRUCTION}\n\nTexto: {texto.strip()}'},
    ]
    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )
    entradas = tokenizer(prompt, return_tensors='pt')
    entradas = {nome: valor.to(DEVICE) for nome, valor in entradas.items()}
    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    novos_tokens = saida[0, entradas['input_ids'].shape[1]:]
    resposta_bruta = tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
    resposta_json = extrair_json(resposta_bruta)
    return {
        'texto': texto,
        'sentimento': resposta_json['sentimento'] if resposta_json else None,
        'json_valido': resposta_json is not None,
        'resposta_bruta': resposta_bruta,
    }

In [12]:
import pandas as pd

exemplos = [
    'O atendimento foi excelente e resolveu meu problema.',
    'A experiência foi frustrante e não recebi nenhuma solução.',
    'Recebi a informação solicitada, sem problemas ou elogios.',
]

resultados = pd.DataFrame(classificar_sentimento(texto) for texto in exemplos)
display(resultados[['texto', 'sentimento', 'json_valido', 'resposta_bruta']])

,texto,sentimento,json_valido,resposta_bruta
0,O atendimento foi excelente e resolveu meu pro...,positivo,True,"{""sentimento"":""positivo""}"
1,A experiência foi frustrante e não recebi nenh...,negativo,True,"{""sentimento"":""negativo""}"
2,"Recebi a informação solicitada, sem problemas ...",positivo,True,"{""sentimento"":""positivo""}"


## 4. Testar uma mensagem digitada pelo usuário

Escolha uma mensagem pronta no campo `Exemplo` para carregá-la automaticamente ou edite o texto livremente. Depois, envie a mensagem para o modelo registrado. O resultado será exibido com um emoji para facilitar a visualização do sentimento identificado.

In [13]:
import html
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

mensagens_exemplo = {
    'Confusa, mas termina feliz': 'No começo achei que seria mais um daqueles atendimentos que prometem resolver tudo e acabam complicando ainda mais. Fiquei esperando, precisei explicar a mesma situação algumas vezes e, por um momento, pensei em desistir. Quando finalmente me responderam, ainda houve uma informação desencontrada, mas depois encontraram o problema, corrigiram o que precisava e me explicaram tudo com calma. Não foi exatamente simples nem rápido, mas no fim consegui o que precisava e saí com a sensação de que valeu a pena ter insistido.',
    'Experiência claramente positiva': 'Fui muito bem atendido, resolveram minha solicitação rapidamente e fiquei satisfeito com o resultado.',
    'Experiência neutra': 'Entrei em contato, recebi as informações solicitadas e o atendimento foi encerrado normalmente.',
    'Atendimento com raiva': 'Estou indignado com essa situação. Já expliquei o problema várias vezes, ninguém resolve nada e continuo sem uma resposta.',
    'Frustração com solução': 'Demorou mais do que deveria e precisei insistir bastante, mas no final resolveram o problema e fiquei aliviado.',
}
seletor_exemplo = widgets.Dropdown(
    options=list(mensagens_exemplo),
    value='Confusa, mas termina feliz',
    description='Exemplo:',
    layout=widgets.Layout(width='100%'),
)
campo_mensagem = widgets.Textarea(
    value=mensagens_exemplo['Confusa, mas termina feliz'],
    placeholder='Digite uma mensagem sobre sua experiência...',
    description='Mensagem:',
    layout=widgets.Layout(width='100%', height='90px'),
)
botao_enviar = widgets.Button(
    description='Enviar mensagem',
    button_style='primary',
    icon='send',
)
saida_interativa = widgets.Output()

emoji_por_sentimento = {
    'negativo': '😞',
    'neutro': '😐',
    'positivo': '😊',
}

def enviar_mensagem(_):
    mensagem = campo_mensagem.value.strip()
    with saida_interativa:
        clear_output()
        if not mensagem:
            display(HTML('<p style="color:#b91c1c">Digite uma mensagem antes de enviar.</p>'))
            return

        resultado = classificar_sentimento(mensagem)
        sentimento = resultado['sentimento'] or 'não identificado'
        emoji = emoji_por_sentimento.get(sentimento, '❓')
        mensagem_segura = html.escape(mensagem)
        resposta_segura = html.escape(resultado['resposta_bruta'])
        display(HTML(
            f'<div style="border:1px solid #d1d5db;border-radius:10px;padding:14px;margin-top:10px;">'
            f'<div style="font-size:32px">{emoji}</div>'
            f'<div><strong>Sentimento identificado:</strong> {html.escape(sentimento)}</div>'
            f'<div><strong>Mensagem:</strong> {mensagem_segura}</div>'
            f'<div><strong>Resposta do modelo:</strong> <code>{resposta_segura}</code></div>'
            '</div>'
        ))

def carregar_exemplo(change):
    if change['name'] == 'value' and change['new'] in mensagens_exemplo:
        campo_mensagem.value = mensagens_exemplo[change['new']]

seletor_exemplo.observe(carregar_exemplo)
botao_enviar.on_click(enviar_mensagem)
display(widgets.VBox([seletor_exemplo, campo_mensagem, botao_enviar, saida_interativa]))

## 5. Registrar o teste de consumo no MLflow

Além de registrar o treinamento e a avaliação, podemos registrar que a versão do adapter foi recuperada do Registry e testada com sucesso. Isso deixa rastreável qual versão foi utilizada e quais exemplos foram executados.

In [14]:
with mlflow.start_run(run_name='teste_modelo_registrado_escutia') as run:
    mlflow.log_params({
        'registered_model': REGISTERED_MODEL,
        'model_version': MODEL_VERSION,
        'model_uri': MODEL_URI,
        'base_model': MODEL_NAME,
    })
    mlflow.log_metrics({
        'test_samples': float(len(resultados)),
        'test_valid_json_rate': float(resultados['json_valido'].mean()),
    })
    mlflow.set_tags({
        'run_type': 'registered_model_test',
        'artifact_type': 'lora_adapter',
    })
    print(f'Teste registrado no MLflow: {run.info.run_id}')

print('Fluxo concluído: versão registrada → adapter baixado → inferência executada.')

Teste registrado no MLflow: 3b08b4d474aa485981dd7b2514acc833
Fluxo concluído: versão registrada → adapter baixado → inferência executada.
